# RAG evaluation, stage 1: retrieval (GPU)

Encodes the RAGTurk chunks and the sampled questions with each model and writes every question's top-5 chunk ids
to Drive (`ragbench/retrieval/<model>.json`). Stage 2 (`colab_generate_judge.ipynb`) needs no GPU: it reads these
files, generates answers with DeepSeek and judges them with GLM. A finished model is skipped on re-run.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
OUT = f'{DRIVE_ROOT}/ragbench'                       # everything produced goes here (never into the repository)
DATA_PATH = f'{DRIVE_ROOT}/data/ragturk/ragturk_formal5k.json'
REPO = 'https://github.com/mericberktas/ragbench'    # set after the repository is pushed
EXTRA = '[retrieve]'

# clone under a name that is NOT `ragbench`: the notebook's working directory is /content, and a folder called
# /content/ragbench would shadow the installed package as an empty namespace package
SRC = '/content/ragbench_src'
if not Path(SRC).exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, SRC], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', SRC + EXTRA], check=True)
# the editable install's .pth file is only read when Python starts, so a running session does not see it:
# put the package folder on the path ourselves
import importlib
if f'{SRC}/src' not in sys.path:
    sys.path.insert(0, f'{SRC}/src')
importlib.invalidate_caches()
import ragbench.data  # fails here, not later, if the package is still not importable

In [ ]:
os.environ['HF_HOME'] = '/content/hf_cache'          # local disk, never Drive
from ragbench.data import load_ragturk, stratified_sample

chunks, all_questions = load_ragturk(DATA_PATH)
Q_PATH = Path(OUT) / 'questions.json'
Q_PATH.parent.mkdir(parents=True, exist_ok=True)
if Q_PATH.exists():
    questions = json.loads(Q_PATH.read_text(encoding='utf-8'))      # the sample is fixed once drawn
else:
    questions = stratified_sample(all_questions, n=500, seed=42)      # at most one question per article
    Q_PATH.write_text(json.dumps(questions, ensure_ascii=False), encoding='utf-8')
print(len(chunks), 'chunks;', len(questions), 'questions')

In [ ]:
INSTR_8K = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:'      # no space after "Query:" (model card)
LAB_INSTR = 'Instruct: Given a Turkish search text, retrieve relevant passages written in Turkish that best match the search text.\nQuery: '
RUN = lambda name: f'{DRIVE_ROOT}/runs/{name}/final'

MODELS = {
    # --- our students ---
    'ours_pair_instr8k_s10800': dict(path=RUN('scale_two_instr8k_s10800_lr1e4/20261027-152516-b2b0c4b')),     # main line (TR-MTEB 67.71)
    'ours_pair_plain_s10800':   dict(path=RUN('scale_two_s10800_lr1e4/20261027-152514-b2b0c4b')),
    'ours_single_bge_s5400':    dict(path=RUN('scale_bge_mix200k_s5400_lr1e4/20261027-101819-77232d7')),
    'ours_single_8k_s5400':     dict(path=RUN('scale_8k_mix200k_s5400_lr1e4/20261027-105836-77232d7')),
    # --- baselines ---
    'raw_base_1k':   dict(hf='ytu-ce-cosmos/modernbert-tr-base-1k', rev='c2e35f553101cce5b03fbb318ef8728473fd34dd'),
    'lab_1k_embed_instr': dict(hf='ytu-ce-cosmos/modernbert-tr-1k-embed', rev='ba1e17bc7e95a257b4e614c300c4ca076711fce6', q=LAB_INSTR),
    'lab_8k_embed_instr': dict(hf='ytu-ce-cosmos/modernbert-tr-embed', rev='855820e0079102805908d60d32cd9fd59c970b80', q=INSTR_8K),
    'bge_m3':        dict(hf='BAAI/bge-m3', rev='5617a9f61b028005a4858fdac845db406aefb181'),
}
for name, spec in MODELS.items():
    if 'path' in spec:
        assert os.path.isdir(spec['path']), f'{name}: {spec["path"]} not found'
print(len(MODELS), 'models')

In [ ]:
from ragbench.retrieve import retrieve_for_model

K = 5
for name, spec in MODELS.items():
    out = retrieve_for_model(name, spec, chunks, questions, Path(OUT) / 'retrieval', k=K, doc_format='content',
                             model_dir=f'/content/models/{name}')
    print(f'{name:<30} gold chunk in top-{K}: {out["gold_hit"]:.3f}   ({out["seconds"]} s)')

In [ ]:
from google.colab import runtime
runtime.unassign()